# 07 - NYC Dispensary Opening Event Study

Within-NYC analysis: does a new cannabis dispensary opening affect 311 complaint patterns in the surrounding zip codes?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

 = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

nyc_311 = pd.read_parquet(Path('../data/processed') / "nyc_311_zip_month.parquet")
disp     = pd.read_parquet(Path('../data/processed') / "nyc_dispensary_events.parquet")
print(f"311 panel: {nyc_311.shape}  |  Zips: {nyc_311['zip'].nunique()}")
print(f"Dispensaries: {len(disp)}")

In [ ]:
DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

nyc_311 = pd.read_parquet(DATA_DIR / "nyc_311_zip_month.parquet")
disp    = pd.read_parquet(DATA_DIR / "nyc_dispensary_events.parquet")
print(f"311 panel: {nyc_311.shape}  |  Dispensaries: {len(disp)}")

## Merge dispensary opening events onto 311 panel

In [ ]:
# For each zip code, find the first dispensary opening date
first_open = disp.groupby('zip')['open_month'].min().reset_index(name='first_open_month')
nyc_311 = nyc_311.merge(first_open, on='zip', how='left')
nyc_311['has_dispensary'] = nyc_311['first_open_month'].notna()
nyc_311['post_open'] = (
    nyc_311['has_dispensary'] &
    (nyc_311['month'] >= nyc_311['first_open_month'])
).astype(float)
nyc_311['rel_month'] = np.where(
    nyc_311['has_dispensary'],
    (pd.to_datetime(nyc_311['month']).dt.to_period('M') -
     pd.to_datetime(nyc_311['first_open_month']).dt.to_period('M')).apply(lambda x: x.n if hasattr(x,'n') else np.nan),
    np.nan
)

WINDOW = 12
print(f"Zip codes with dispensary: {nyc_311['has_dispensary'].groupby(nyc_311['zip']).first().sum()}")
print(f"Zip codes without: {(~nyc_311['has_dispensary']).groupby(nyc_311['zip']).first().sum()}")

## Event study: 311 complaints around dispensary opening

In [ ]:
from linearmodels.panel import PanelOLS

outcome = 'log_total_complaints'
ALL_PERIODS = [k for k in range(-WINDOW, WINDOW+1) if k != -1]

for k in ALL_PERIODS:
    nyc_311[f"dum_{k}"] = (
        nyc_311['has_dispensary'] &
        (nyc_311['rel_month'].fillna(9999).clip(-WINDOW,WINDOW) == k)
    ).astype(float)

dum_cols = [f"dum_{k}" for k in ALL_PERIODS]
nyc_idx = nyc_311.set_index(['zip','month'])

es_res = PanelOLS(
    nyc_idx[outcome],
    nyc_idx[dum_cols],
    entity_effects=True,
    time_effects=True,
).fit(cov_type='clustered', cluster_entity=True)

es_df = pd.DataFrame({
    'rel_month': ALL_PERIODS,
    'coef':  es_res.params[dum_cols].values,
    'ci_lo': es_res.conf_int().loc[dum_cols,'lower'].values,
    'ci_hi': es_res.conf_int().loc[dum_cols,'upper'].values,
}).sort_values('rel_month')

fig, ax = plt.subplots(figsize=(12,5))
ax.fill_between(es_df['rel_month'], es_df['ci_lo'], es_df['ci_hi'], alpha=0.2, c='steelblue')
ax.plot(es_df['rel_month'], es_df['coef'], 'o-', ms=4, lw=1.5, c='steelblue')
ax.plot(-1, 0, 'ro', ms=8, zorder=5, label='Reference (t=-1)')
ax.axvline(0, color='red', ls='--', lw=1.5, label='Dispensary opens')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel("Months Relative to First Dispensary in Zip Code")
ax.set_ylabel("Coefficient on log 311 Complaints")
ax.set_title("NYC: Effect of Cannabis Dispensary Opening on 311 Complaints\nTWFE; clustered SEs (zip); window = ±12 months")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "07_nyc_dispensary_event_study.png", bbox_inches='tight')
plt.show()